# 01 · Collecte et Ingestion des Données
## CEET Smart Grid – Energy Blackout Prediction
**Objectif :** Charger, inspecter et comprendre le dataset du réseau électrique togolais.

In [1]:
import sys, os
sys.path.insert(0, '../src')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Chemins
RAW = Path('../data/raw/ceet_togo_smartgrid_dataset.csv')
print(f"Fichier : {RAW}")
print(f"Taille  : {RAW.stat().st_size / 1e6:.2f} Mo")

Fichier : ..\data\raw\ceet_togo_smartgrid_dataset.csv
Taille  : 8.40 Mo


### 1.1 Chargement du Dataset

In [2]:
df = pd.read_csv(RAW, parse_dates=['datetime'])
print(f"Shape   : {df.shape}")
print(f"Colonnes: {df.columns.tolist()}")
df.head(3)

Shape   : (50000, 25)
Colonnes: ['datetime', 'region', 'city', 'temperature', 'humidity', 'season', 'hour', 'industrial_load', 'residential_load', 'commercial_load', 'total_load_mw', 'available_power_mw', 'renewable_power_mw', 'transformer_temp', 'voltage', 'frequency', 'event', 'outage_risk', 'overload', 'blackout', 'load_shedding_mw', 'energy_price', 'fuel_cost', 'population_density', 'smart_meter_count']


,datetime,region,city,temperature,humidity,season,hour,industrial_load,residential_load,commercial_load,...,frequency,event,outage_risk,overload,blackout,load_shedding_mw,energy_price,fuel_cost,population_density,smart_meter_count
0,2020-01-01 00:00:00,Lome,Sokode,34.99,63.87,Seche,0,129.08,104.70,73.07,...,49.59,Match de Football,100.00,1,0,27.35,108.97,581.41,761,6713
1,2020-01-01 01:00:00,Lome,Lome,41.14,40.05,Seche,1,102.76,72.78,62.52,...,49.95,Event Politique,91.47,1,0,13.83,83.23,732.44,2245,8388
2,2020-01-01 02:00:00,Centrale,Kpalime,42.05,49.02,Seche,2,137.22,65.28,56.81,...,50.47,Pas evenement,100.00,1,1,14.57,122.78,1285.61,1808,1494


In [3]:
print("=== Types de données ===")
print(df.dtypes)
print()
print("=== Statistiques descriptives ===")
df.describe().round(2)

=== Types de données ===
datetime              datetime64[ns]
region                        object
city                          object
temperature                  float64
humidity                     float64
season                        object
hour                           int64
industrial_load              float64
residential_load             float64
commercial_load              float64
total_load_mw                float64
available_power_mw           float64
renewable_power_mw           float64
transformer_temp             float64
voltage                      float64
frequency                    float64
event                         object
outage_risk                  float64
overload                       int64
blackout                       int64
load_shedding_mw             float64
energy_price                 float64
fuel_cost                    float64
population_density             int64
smart_meter_count              int64
dtype: object

=== Statistiques descriptives ===


,datetime,temperature,humidity,hour,industrial_load,residential_load,commercial_load,total_load_mw,available_power_mw,renewable_power_mw,...,voltage,frequency,outage_risk,overload,blackout,load_shedding_mw,energy_price,fuel_cost,population_density,smart_meter_count
count,50000,50000.00,50000.00,50000.00,50000.00,50000.00,50000.00,50000.00,50000.00,50000.00,...,50000.00,50000.00,50000.00,50000.00,50000.00,50000.00,50000.00,50000.00,50000.00,50000.00
mean,2022-11-07 15:30:00,30.21,67.33,11.50,119.96,89.99,59.90,310.33,260.02,22.55,...,219.94,50.00,94.61,0.82,0.04,14.37,139.70,999.31,2522.47,5038.61
min,2020-01-01 00:00:00,19.96,40.00,0.00,2.44,-2.82,-7.43,117.63,178.19,5.00,...,189.32,48.72,50.40,0.00,0.00,0.00,80.00,500.00,50.00,100.00
25%,2021-06-04 19:45:00,27.28,53.44,5.00,99.80,70.51,49.70,275.82,246.51,13.76,...,214.59,49.80,90.66,1.00,0.00,7.21,109.58,750.23,1289.00,2565.00
50%,2022-11-07 15:30:00,29.17,67.19,11.00,120.10,86.37,59.94,308.47,260.02,22.52,...,219.94,50.00,99.37,1.00,0.00,14.74,139.60,997.94,2508.00,5036.00
75%,2024-04-11 11:15:00,32.82,81.19,17.00,140.09,105.20,70.05,343.00,273.49,31.40,...,225.33,50.20,100.00,1.00,0.00,22.37,169.84,1248.62,3754.00,7520.25
max,2025-09-14 07:00:00,47.08,95.00,23.00,256.84,221.59,122.31,551.42,341.67,40.00,...,251.52,51.43,100.00,1.00,1.00,30.00,199.99,1499.95,5000.00,10000.00
std,NaN,4.03,15.93,6.92,29.94,28.58,15.08,50.62,19.94,10.13,...,7.96,0.30,7.52,0.38,0.19,9.33,34.78,287.97,1428.35,2858.59


### 1.2 Analyse de la Qualité des Données

In [4]:
# Valeurs manquantes
missing = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(2)
missing_df = pd.DataFrame({'Manquantes': missing, 'Pourcentage (%)': missing_pct})
missing_df[missing_df['Manquantes'] > 0]

,Manquantes,Pourcentage (%)


In [5]:
# Doublons
n_dup = df.duplicated().sum()
print(f"Doublons : {n_dup}")

# Distribution des cibles
print("\n=== Distribution BLACKOUT ===")
print(df['blackout'].value_counts(normalize=True).round(3))
print("\n=== Distribution OVERLOAD ===")
print(df['overload'].value_counts(normalize=True).round(3))

Doublons : 0

=== Distribution BLACKOUT ===
blackout
0    0.96
1    0.04
Name: proportion, dtype: float64

=== Distribution OVERLOAD ===
overload
1    0.825
0    0.175
Name: proportion, dtype: float64


### 1.3 Aperçu des Variables Catégorielles

In [6]:
cat_cols = ['region', 'city', 'season', 'event']
for col in cat_cols:
    print(f"\n{col.upper()} ({df[col].nunique()} valeurs uniques):")
    print(df[col].value_counts().head(8))


REGION (6 valeurs uniques):
region
Plateaux    8509
Centrale    8397
Savanes     8351
Maritime    8270
Lome        8237
Kara        8236
Name: count, dtype: int64

CITY (7 valeurs uniques):
city
Dapaong     7258
Tsevie      7213
Kpalime     7193
Atakpame    7164
Lome        7085
Sokode      7045
Kara        7042
Name: count, dtype: int64

SEASON (2 valeurs uniques):
season
Pluvieuse    34136
Seche        15864
Name: count, dtype: int64

EVENT (5 valeurs uniques):
event
Pas evenement        10065
Event Politique      10046
Concert               9979
Match de Football     9964
Festivale             9946
Name: count, dtype: int64


In [7]:
# Plage temporelle
print(f"Période  : {df['datetime'].min()} → {df['datetime'].max()}")
print(f"Durée    : {(df['datetime'].max() - df['datetime'].min()).days} jours")
print(f"Fréquence: Horaire ({df.shape[0]} lectures)")

Période  : 2020-01-01 00:00:00 → 2025-09-14 07:00:00
Durée    : 2083 jours
Fréquence: Horaire (50000 lectures)


### 1.4 Sauvegarde du Dataset Brut (copie de travail)

In [8]:
import shutil
dest = Path('../data/raw/ceet_togo_smartgrid_dataset.csv')
print(f"Dataset brut disponible : {dest.exists()}")
print("\n Collecte terminée prêt pour le nettoyage (02_data_cleaning.ipynb)")

Dataset brut disponible : True

 Collecte terminée prêt pour le nettoyage (02_data_cleaning.ipynb)
